# Imputation evaluation (BraSyn metrics)

Evaluates the output of `imputation/run_imputation.py`: N imputed runs of one dropped modality per subject.
All metrics come from the [BraSyn tutorial](https://github.com/WinstonHuTiger/BraSyn_tutorial) code, and every
volume is prepared exactly as its `project/test.py` does (IPL orientation, 144×192×192 crop, min-max to [0, 1]).

Setup, once, from the repo root:
```
git clone https://github.com/WinstonHuTiger/BraSyn_tutorial external/BraSyn_tutorial
```
Run the notebook from the `trust_pipeline/` folder.

In [ ]:
import sys
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import torch
from skimage.metrics import peak_signal_noise_ratio as psnr  # what BraSyn's test.py imports

BRASYN_REPO = Path("../external/BraSyn_tutorial")
IMPUTED_DIR = Path("../results/W_preprocess_imputed_multiple_runs")  # output_dir in imputation/config.yaml
MOD = "t1c"                                                           # the imputed modality to evaluate
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"               # full-size 3D SSIM on CPU can run out of RAM

sys.path.insert(0, str(BRASYN_REPO / "project"))
from util.ssim import ssim                                                 # BraSyn's 3D SSIM
from data.data_augmentation_3D import getBetterOrientation, toGrayScale    # BraSyn's preprocessing

## Load one subject

`load` is BraSyn's dataloader transform (`data/brain_3D_random_mod_dataset.py`): reorient to IPL, then crop.
The real image is also rescaled to [0, 1] with `toGrayScale`; the imputed runs already are (they are BraSyn's output).

In [ ]:
def load(path):
    return getBetterOrientation(nib.load(path), "IPL").get_fdata()[8:152, 24:216, 24:216]


def load_subject(mod_dir):
    real = toGrayScale(load(next((mod_dir / "original").glob("*.nii*")))).astype(np.float32)
    imputed = np.stack([load(p) for p in sorted((mod_dir / "imputed").glob("*.nii*"))]).astype(np.float32)
    return real, imputed


mod_dirs = sorted(IMPUTED_DIR.glob(f"*/*/{MOD}"))   # <class>/<subject>/<mod>
mod_dir = mod_dirs[0]                                # change the index to pick another subject
real, imputed = load_subject(mod_dir)
print(mod_dir.parent.name, "| real", real.shape, "| imputed", imputed.shape, "(N runs, I, P, L)")

In [ ]:
z = real.shape[0] // 2   # middle axial slice (axis 0 is inferior-superior in IPL)
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (title, img) in zip(axes, {"real": real, "imputed (run 0)": imputed[0], "std across runs": imputed.std(0)}.items()):
    ax.imshow(img[z], cmap="gray")
    ax.set_title(title)
    ax.axis("off")

## 1a. Imputed vs real

SSIM and PSNR of each run against the real image, as in BraSyn's `test.py`
(SSIM over the whole cropped volume, PSNR with `data_range` = range of the imputed image).

In [ ]:
def t(x):
    return torch.from_numpy(x)[None, None].to(DEVICE)   # (1, 1, I, P, L), the shape BraSyn's ssim expects


def compare_to_real(real, imputed):
    return pd.DataFrame({
        "SSIM": [ssim(t(real), t(run)).item() for run in imputed],
        "PSNR": [psnr(real, run, data_range=run.max() - run.min()) for run in imputed],
    })


per_run = compare_to_real(real, imputed)
per_run.agg(["mean", "std"]).T

## 1b. Spread across the N runs (no ground truth needed)

How different the runs are from each other: voxel-wise std inside the brain, and SSIM between every pair of runs.

In [ ]:
def spread(real, imputed):
    pairwise = [ssim(t(a), t(b)).item() for a, b in combinations(imputed, 2)]
    return {"voxel std (brain)": imputed.std(0)[real > 0].mean(),
            "pairwise SSIM mean": np.mean(pairwise), "pairwise SSIM std": np.std(pairwise)}


pd.Series(spread(real, imputed))

## All subjects → CSV

In [ ]:
rows = []
for d in mod_dirs:
    real, imputed = load_subject(d)
    rows.append({"subject": d.parent.name, "n_runs": len(imputed),
                 **compare_to_real(real, imputed).mean().add_suffix(" (mean over runs)"), **spread(real, imputed)})
results = pd.DataFrame(rows).set_index("subject")
results.to_csv(f"metrics_imputation_{MOD}.csv")
results

## 2. Segmentation — to do

Needs one mask per (imputed run n, prompt k). Plan: official BraTS metrics with
`brats_evaluation.evaluate_single_exam` (`pip install BraTS-evaluation`, config `gli`).

## 3. Imputation × segmentation — to do

Plan: table of Dice per (n, k) → `statsmodels` `anova_lm(ols("dice ~ C(n) + C(k)", df).fit())`
to split the variability between imputation and prompt.